In [ ]:
import json
import matplotlib.pyplot as plt
import pandas as pd
import re
import seaborn as sns

from dotenv import load_dotenv
from matplotlib.patches import Patch
from openinference.semconv.trace import (
    OpenInferenceSpanKindValues,
    SpanAttributes
)
from tqdm.notebook import tqdm

load_dotenv()

SPAN_KIND = SpanAttributes.OPENINFERENCE_SPAN_KIND
OUTPUT_VALUE = SpanAttributes.OUTPUT_VALUE
AGENT = OpenInferenceSpanKindValues.AGENT.value

In [ ]:
original_input_path = "../../data/frames/llm_frames_results_all_ctx_40960_kvq_f16.csv"
original_input = pd.read_csv(original_input_path)
questions = original_input["Prompt"]
answers = original_input["Answer"]

base_path = "../../logs/frames_ollama_qwen3_8b"
data = []
for i in range(150):
    path = f"{base_path}/{i}"
    for run_idx in range(1):
        run_path = f"{path}/run_{run_idx}"
        summary_path = f"{run_path}/analysis/summary.json"
        step_summary_path = f"{run_path}/analysis/step_summary.json"

        with open(summary_path, "r") as f:
            summary = json.load(f)
        with open(step_summary_path, "r") as f:
            step_summary = json.load(f)

        step_data = {}
        step_idx = 0
        for step in step_summary:
            if step["kind"] != "LLM":
                continue
            for k, v in step.items():
                step_data[f"step_{step_idx}_{k}"] = v
            step_idx += 1

        agent_output = summary["agent_output"]
        if agent_output is not None and "<think>" in agent_output and "</think>" in agent_output:
            summary["agent_output"] = agent_output[agent_output.find("</think>")+len("</think>"):].strip()

        data.append({
            "question": questions[i],
            "answer": answers[i],
            **summary,
            **step_data,
        })

data = pd.DataFrame.from_dict(data)
data.to_csv("../../data/frames/frames_profile_results_qwen3_8b.csv", index=False)

In [ ]:
data = pd.read_csv("../../data/frames/frames_profile_results_qwen3_fixed_cascade_compress_1.7b_to_14b_step_1_judged.csv")
# data = pd.read_csv("../../data/frames/frames_profile_results_qwen3_1.7b_judged.csv")
# data.groupby("agent_output_eval")["energy_total_mWh"].quantile([0.0, 0.05, 0.25, 0.5, 0.75, 0.95, 1.0])
# data["energy_total_mWh"].quantile([0.0, 0.05, 0.25, 0.5, 0.75, 0.95, 1.0])
# data["energy_total_mWh"].hist()
# data["energy_total_mWh"].idxmax()
# data.boxplot("energy_total_mWh")
# data[data["num_LLM_calls"] == 11][["agent_output", "energy_total_mWh"]]
# data["num_LLM_calls"].hist()
# data["agent_output_eval"].value_counts()

In [ ]:
num_calls = sorted(data["num_LLM_calls"].unique())
plt.boxplot([data[data["num_LLM_calls"] == n]["energy_total_mWh"] for n in num_calls], labels=num_calls)
plt.xlabel("Num LLM calls")
plt.ylabel("Total energy used (mWh)")
plt.show()

In [ ]:
agent_output_labels = sorted(data["agent_output_eval"].unique())
plt.boxplot([data[data["agent_output_eval"] == l]["energy_total_mWh"] for l in agent_output_labels], labels=agent_output_labels)
plt.xlabel("Agent output")
plt.ylabel("Total energy used (mWh)")
plt.show()

In [ ]:
acc = []
energy_usage = []

inputs = [
    "../../data/frames/frames_profile_results_qwen3_1.7b_judged.csv",
    "../../data/frames/frames_profile_results_qwen3_4b_judged.csv",
    "../../data/frames/frames_profile_results_qwen3_8b_judged.csv",
    "../../data/frames/frames_profile_results_qwen3_fixed_cascade_compress_1.7b_to_14b_step_1_judged.csv",
    "../../data/frames/frames_profile_results_qwen3_14b_judged.csv",
]

for path in inputs:
    data = pd.read_csv(path)
    acc.append(round((data["agent_output_eval"] == "CORRECT").mean(), 2))
    # median_energy.append(data["energy_total_mWh"].median())
    energy_usage.append(data["energy_total_mWh"])

In [ ]:
labels = ["1.7B", "4B", "8B", "Comcade 1.7B to 14B", "14B"]
colors = sns.color_palette("Set2", n_colors=len(labels))

plt.figure(figsize=(8, 5))
handles = []

for i in range(len(energy_usage)):
    bp = plt.boxplot(
        [energy_usage[i]],
        vert=False,
        positions=[acc[i]],
        widths=0.025,
        showfliers=False,
        patch_artist=True
    )

    # Only color the box body
    box = bp['boxes'][0]
    box.set_facecolor(colors[i])
    box.set_linewidth(1)  # thinner border
    # Customize median line
    median = bp['medians'][0]
    median.set_color('black')

    # Add to legend
    handles.append(Patch(facecolor=colors[i], label=labels[i]))

# Final formatting
plt.ylim(max(min(acc) - 0.1, 0.0), min(max(acc) + 0.1, 1.0))
plt.grid(True, axis='x')
plt.xlabel("Energy Usage (mWh)")
plt.ylabel("Accuracy")
plt.title("Accuracy vs Energy Usage on FRAMES")
plt.legend(handles=handles, title="Models")
plt.tight_layout()
plt.show()